# Hafta 6 — Sınıflandırma I

Lojistik regresyon, k-NN, karışıklık matrisi, ROC ve eşik seçimi. Veri: `motor_ariza.csv`.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (confusion_matrix, accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, roc_curve, classification_report, log_loss)

m = pd.read_csv("motor_ariza.csv")
ozellik = ["calisma_saati", "titresim_rms_mms", "titresim_kurtosis", "sicaklik_C", "akim_dengesizlik_pct"]
X, y = m[ozellik], m["ariza"]
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)
print(m.shape, "arıza oranı:", y.mean().round(3)); m.head()

## 1. Sigmoid ve log-loss: elle (Örnek 6.1, 6.2)

In [ ]:
sigmoid = lambda z: 1/(1 + np.exp(-z))
z = 1.8*1.2 + 1.1*0.4 - 0.5
print("z =", round(z, 2), " p =", round(sigmoid(z), 3), " odds =", round(sigmoid(z)/(1-sigmoid(z)), 2))

y3 = np.array([1, 0, 1]); p3 = np.array([0.9, 0.2, 0.3])
kayip = -(y3*np.log(p3) + (1-y3)*np.log(1-p3))
print("kayıplar:", kayip.round(3), " ortalama:", kayip.mean().round(3), " sklearn:", round(log_loss(y3, p3), 3))

## 2. Keşif: özellikler sınıfları ayırıyor mu?

In [ ]:
fig, ax = plt.subplots(1, 5, figsize=(15, 3))
for a, k in zip(ax, ozellik):
    a.hist(m.loc[m.ariza == 0, k], bins=30, alpha=.6, label="sağlam"); a.hist(m.loc[m.ariza == 1, k], bins=30, alpha=.6, label="arızalı"); a.set_title(k, fontsize=9)
ax[0].legend(); plt.tight_layout(); plt.show()

**Soru:** Hangi özellik iki sınıfı en iyi ayırıyor? Hangisi neredeyse hiç ayırmıyor?

## 3. Lojistik regresyon

In [ ]:
lojistik = make_pipeline(StandardScaler(), LogisticRegression()).fit(Xtr, ytr)
p_lr = lojistik.predict_proba(Xte)[:, 1]
tahmin = (p_lr >= 0.5).astype(int)
cm = confusion_matrix(yte, tahmin); print("Karışıklık matrisi [[TN FP],[FN TP]]:"); print(cm)
print(classification_report(yte, tahmin, digits=3))
print("AUC:", round(roc_auc_score(yte, p_lr), 3), " log-loss:", round(log_loss(yte, p_lr), 3))

## 4. Ölçütleri elle doğrulayalım (Örnek 6.4)

In [ ]:
TN, FP, FN, TP = cm.ravel()
print(f"TP={TP} FN={FN} FP={FP} TN={TN}")
print("doğruluk :", round((TP+TN)/(TP+TN+FP+FN), 3), round(accuracy_score(yte, tahmin), 3))
print("precision:", round(TP/(TP+FP), 3), round(precision_score(yte, tahmin), 3))
print("recall   :", round(TP/(TP+FN), 3), round(recall_score(yte, tahmin), 3))
P, R = TP/(TP+FP), TP/(TP+FN); print("F1       :", round(2*P*R/(P+R), 3), round(f1_score(yte, tahmin), 3))
print("'hepsi sağlam' doğruluğu:", round((yte == 0).mean(), 3))

## 5. Katsayıları yorumlamak

Standartlaştırılmış özelliklerde katsayı: 'bir σ artış log-odds'u kaç birim artırır'; e^w = odds çarpanı.

In [ ]:
w = lojistik[-1].coef_[0]
print(pd.DataFrame({"katsayı (log-odds/σ)": w.round(3), "odds çarpanı e^w": np.exp(w).round(2)}, index=ozellik).sort_values("katsayı (log-odds/σ)", ascending=False))

## 6. k-NN: ölçeklemenin ve k'nın etkisi

In [ ]:
# Ölçeklemeden
knn_olceksiz = KNeighborsClassifier(15).fit(Xtr, ytr)
print("Ölçeksiz k-NN  AUC:", round(roc_auc_score(yte, knn_olceksiz.predict_proba(Xte)[:, 1]), 3))
# Ölçekli, k taraması (5-katlı stratified CV ile)
cv = StratifiedKFold(5, shuffle=True, random_state=0)
sonuc = []
for k in (1, 3, 5, 9, 15, 25, 51, 101):
    pipe = make_pipeline(StandardScaler(), KNeighborsClassifier(k))
    s = cross_val_score(pipe, Xtr, ytr, cv=cv, scoring="roc_auc"); sonuc.append((k, s.mean(), s.std()))
    print(f"k={k:3d}  CV AUC = {s.mean():.3f} ± {s.std():.3f}")

In [ ]:
ks = [r[0] for r in sonuc]; plt.errorbar(ks, [r[1] for r in sonuc], [r[2] for r in sonuc], marker="o"); plt.xscale("log"); plt.xlabel("k"); plt.ylabel("CV AUC"); plt.grid(alpha=.3); plt.show()
en_iyi_k = max(sonuc, key=lambda r: r[1])[0]; print("en iyi k:", en_iyi_k)
knn = make_pipeline(StandardScaler(), KNeighborsClassifier(en_iyi_k)).fit(Xtr, ytr); p_knn = knn.predict_proba(Xte)[:, 1]
print("Test AUC k-NN:", round(roc_auc_score(yte, p_knn), 3), " lojistik:", round(roc_auc_score(yte, p_lr), 3))

## 7. ROC eğrisi

In [ ]:
plt.figure(figsize=(5, 4.5))
for ad, pp in (("lojistik", p_lr), ("k-NN", p_knn)):
    fpr, tpr, _ = roc_curve(yte, pp); plt.plot(fpr, tpr, label=f"{ad} AUC={roc_auc_score(yte, pp):.3f}")
plt.plot([0, 1], [0, 1], "k--", label="rastgele"); plt.xlabel("FPR"); plt.ylabel("TPR (recall)"); plt.legend(); plt.grid(alpha=.3); plt.show()

## 8. Maliyete göre eşik (Örnek 6.6)

FN = 10 000 TL, FP = 500 TL. Eşiği doğrulama kümesinde seçmek gerekir; burada öğretim için test kümesi üzerinde tarıyoruz.

In [ ]:
def maliyet(y, p, esik, c_fn=10000, c_fp=500):
    tn, fp, fn, tp = confusion_matrix(y, p >= esik).ravel()
    return c_fn*fn + c_fp*fp, tp, fn, fp, tn
esikler = np.arange(0.05, 0.96, 0.05); mal = [maliyet(yte, p_lr, e)[0] for e in esikler]
for e in (0.15, 0.3, 0.5, 0.7):
    c, tp, fn, fp, tn = maliyet(yte, p_lr, e); print(f"eşik {e:.2f}: maliyet {c:>8,} TL   TP={tp} FN={fn} FP={fp} TN={tn}  doğruluk={(tp+tn)/len(yte):.3f}")
plt.plot(esikler, mal, "o-"); plt.xlabel("eşik"); plt.ylabel("toplam maliyet (TL)"); plt.grid(alpha=.3); plt.show()
print("en iyi eşik:", esikler[int(np.argmin(mal))].round(2))

## 9. Alıştırmalar

**Alıştırma 1.** z = −1.5, 0, 0.7, 3 için sigmoid değerlerini ve eşik 0.5 ile 0.2'de verilen sınıfları tablo hâlinde yazdırın.

In [ ]:
# Alıştırma 1

**Alıştırma 2.** `class_weight='balanced'` ile lojistik regresyonu yeniden eğitin. Precision, recall, F1 ve AUC nasıl değişti? Neden?

In [ ]:
# Alıştırma 2

**Alıştırma 3.** Yalnızca `titresim_rms_mms` ile lojistik regresyon eğitin; karar sınırının (p = 0.5) hangi RMS değerine denk geldiğini katsayılardan hesaplayın (ipucu: z = 0 → x = −b/w, ölçeklemeyi geri alın). ISO 10816'daki 4.5 mm/s 'alarm' sınırıyla karşılaştırın.

In [ ]:
# Alıştırma 3

**Alıştırma 4.** ROC eğrisini kendiniz çizin: eşiği 0'dan 1'e 0.02 adımla tarayıp her adımda FPR ve TPR hesaplayın; `roc_curve` ile karşılaştırın. AUC'yi yamuk kuralıyla (`np.trapz`) hesaplayın.

In [ ]:
# Alıştırma 4

**Alıştırma 5.** k-NN'i `weights='distance'` ile eğitin (yakın komşu daha çok oy). CV AUC değişti mi? Hangi k'da fark daha büyük, neden?

In [ ]:
# Alıştırma 5